# 03b  -  Run `ai_query` (token-based pricing path)

This notebook runs `ai_query` on the same transcripts from notebook 02, using the same Foundation Model (default: Llama-3.3-70B). It logs a token-based predicted cost and the run window, so notebook 04 can compare the two paths side-by-side.

**Why `ai_query`:** the `v1` AI Functions (`ai_query`, `ai_summarize`, `ai_translate`, `ai_gen`, `ai_mask`, `ai_fix_grammar`, `ai_analyze_sentiment`) run on Foundation Model Serving. Pricing is published per million tokens, but in [`system.billing.usage`](https://docs.databricks.com/en/admin/system-tables/billing.html) it shows up as `usage_type = 'COMPUTE_TIME'` DBUs under `billing_origin_product = 'MODEL_SERVING'` with `offering_type = 'BATCH_INFERENCE'` (token counts are abstracted at billing-table level).

**What this notebook does:**
1. Reads job parameters.
2. Predicts cost using the **token-based** formula from the [Foundation Model APIs pricing page](https://www.databricks.com/product/pricing/foundation-model-serving).
3. Calls `ai_query` with a structured-JSON prompt asking for the same four fields as `03a`.
4. Captures the run window and logs everything to `run_log`.

**Public references:**
- [`ai_query` SQL function reference](https://docs.databricks.com/en/sql/language-manual/functions/ai_query.html)
- [Foundation Model APIs pricing](https://www.databricks.com/product/pricing/foundation-model-serving)
- [Proprietary FM serving pricing (Claude, GPT, Gemini)](https://www.databricks.com/product/pricing/proprietary-foundation-model-serving)
- [Cost queries for AI Functions on Azure](https://learn.microsoft.com/en-us/azure/databricks/large-language-models/ai-functions#view-costs-for-ai-function-workloads)


## Read job parameters


In [ ]:
import datetime as dt, uuid

# Helper: timezone-aware UTC, then strip tz to keep parity with Spark's naive
# TimestampType columns. Replaces the deprecated dt.datetime.utcnow().
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

dbutils.widgets.text("catalog",      "main")
dbutils.widgets.text("schema",       "ai_functions_cost_demo")
dbutils.widgets.text("sample_size",  "100")
dbutils.widgets.text("team_tag",     "ai-cost-demo")
dbutils.widgets.text("owner_email",  "you@example.com")
dbutils.widgets.text("model_name",   "databricks-meta-llama-3-3-70b-instruct")

CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")
TEAM_TAG    = dbutils.widgets.get("team_tag")
OWNER_EMAIL = dbutils.widgets.get("owner_email")
MODEL_NAME  = dbutils.widgets.get("model_name")

TRANSCRIPTS_TABLE = f"{CATALOG}.{SCHEMA}.transcripts"
RESULTS_TABLE     = f"{CATALOG}.{SCHEMA}.query_results"
RUN_LOG_TABLE     = f"{CATALOG}.{SCHEMA}.run_log"

RUN_ID = "ai_query-" + utc_now().strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
print(f"Run id:        {RUN_ID}")
print(f"Model:         {MODEL_NAME}")
print(f"Reads from:    {TRANSCRIPTS_TABLE}")
print(f"Writes to:     {RESULTS_TABLE}")


## Predict the cost  -  token-based formula

The next cell computes the predicted cost using the per-token formula from the [Foundation Model APIs pricing page](https://www.databricks.com/product/pricing/foundation-model-serving).

```
input_tokens  ~= chars / 4
output_tokens ~= rows * 80   (4 fields x ~20 tokens each, in JSON)
cost_usd       = (input_tokens / 1M) x $/M_input
              + (output_tokens / 1M) x $/M_output
```

**Llama-3.3-70B Azure list rates (validated 2026-04-28):**
- Input:  7.143 DBUs/M tokens × \$0.07/DBU = **\$0.50 / M input tokens**
- Output: 21.429 DBUs/M tokens × \$0.07/DBU = **\$1.50 / M output tokens**

Note: this prediction is the *list-price* concept. The actual amount in `system.billing.usage` is recorded as DBU compute time, not raw token counts  -  but the underlying economics are the same.


In [ ]:
from pyspark.sql import functions as F

# Token-based list rates for Llama-3.3-70B on Azure (the default model)
USD_PER_M_INPUT_LLAMA  = 0.50
USD_PER_M_OUTPUT_LLAMA = 1.50

# If a different model is configured, use its rates. Lookup table validated 2026-04-28
# from https://www.databricks.com/product/pricing/foundation-model-serving and
# https://www.databricks.com/product/pricing/proprietary-foundation-model-serving
MODEL_RATES = {
    "databricks-meta-llama-3-3-70b-instruct":  (0.50, 1.50),
    "databricks-meta-llama-3-1-8b-instruct":   (0.15, 0.45),
    "databricks-gpt-oss-120b":                 (0.15, 0.60),
    "databricks-claude-sonnet-4":              (4.50, 22.50),
    "databricks-claude-opus-4":                (7.50, 37.50),
}
in_rate, out_rate = MODEL_RATES.get(MODEL_NAME, (USD_PER_M_INPUT_LLAMA, USD_PER_M_OUTPUT_LLAMA))

stats = (
    spark.table(TRANSCRIPTS_TABLE)
        .select(
            F.count("*").alias("rows"),
            F.sum(F.length("transcript")).alias("total_chars"),
        ).collect()[0]
)
n_rows      = stats["rows"]
total_chars = stats["total_chars"]

input_tokens  = total_chars / 4.0
output_tokens = n_rows * 80         # ~20 tokens per JSON field x 4 fields
predicted_cost = (input_tokens / 1_000_000.0) * in_rate + (output_tokens / 1_000_000.0) * out_rate

print(f"Model:                 {MODEL_NAME}")
print(f"  $/M input tokens:    ${in_rate:.2f}")
print(f"  $/M output tokens:   ${out_rate:.2f}")
print()
print(f"Rows:                  {n_rows:,}")
print(f"Estimated input toks:  {input_tokens:,.0f}")
print(f"Estimated output toks: {output_tokens:,.0f}")
print()
print(f"PREDICTED ai_query cost (Azure list): ${predicted_cost:.4f}")
print(f"  Per row:                            ${predicted_cost / n_rows:.6f}")

PRED = {
    "n_rows": n_rows, "total_chars": int(total_chars),
    "input_tokens": int(input_tokens), "output_tokens": int(output_tokens),
    "predicted_cost": predicted_cost,
    "input_rate": in_rate, "output_rate": out_rate,
}


## Run `ai_query` and capture the run window

The next cell calls `ai_query` with a JSON-output prompt asking for the same four fields as `ai_extract`. Using a structured prompt keeps outputs comparable between the two paths.

We use:
- `failOnError => false` so a single bad row doesn't abort the batch (the function returns `STRUCT<result, errorMessage>`).
- `modelParameters => named_struct('max_tokens', 200, 'temperature', 0.0)` for deterministic, bounded output.
- A SQL `WITH` block to unpack the STRUCT into separate `ai_output` (STRING) and `ai_error` (STRING) columns  -  keeps downstream parsing simple.


In [ ]:
from pyspark.sql import functions as F

PROMPT_INSTRUCTION = (
    "You are a fraud detection assistant. From the call transcript below, "
    "extract these fields and return ONLY a JSON object: "
    '{"caller_name": "...", "account_id": "...", "dollar_amount": "...", "fraud_signal_phrase": "..."}'
    " Use empty string if a field is not present.\n\nTRANSCRIPT:\n"
)

run_start = utc_now()
print(f"Run start (UTC): {run_start.isoformat()}")

# Build the prompt column on the dataframe, then ai_query in SQL
prompted = (
    spark.table(TRANSCRIPTS_TABLE)
        .withColumn("prompt", F.concat(F.lit(PROMPT_INSTRUCTION), F.col("transcript")))
)
prompted.createOrReplaceTempView("transcripts_prompted")

query_sql = f"""
    WITH inferred AS (
        SELECT
            transcript_id, caller_name_truth, account_id_truth, dollar_amount_truth,
            is_fraud_truth, fraud_phrase_truth,
            ai_query(
                '{MODEL_NAME}',
                prompt,
                modelParameters => named_struct('max_tokens', 200, 'temperature', 0.0),
                failOnError => false
            ) AS ai_response
        FROM transcripts_prompted
    )
    SELECT
        transcript_id, caller_name_truth, account_id_truth, dollar_amount_truth,
        is_fraud_truth, fraud_phrase_truth,
        COALESCE(ai_response.result, '')          AS ai_output,
        ai_response.errorMessage                  AS ai_error,
        current_timestamp()                       AS run_ts
    FROM inferred
"""

result_df = (
    spark.sql(query_sql)
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("team",   F.lit(TEAM_TAG))
        .withColumn("path",   F.lit("ai_query"))
)

(result_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RESULTS_TABLE))

run_end = utc_now()
duration_sec = (run_end - run_start).total_seconds()
print(f"Run end   (UTC): {run_end.isoformat()}")
print(f"Duration:        {duration_sec:.1f} sec  ({duration_sec/60:.2f} min)")

display(spark.table(RESULTS_TABLE).select("transcript_id", "ai_output", "fraud_phrase_truth").limit(5))


## Log the run for reconciliation

Same `run_log` table as `03a`  -  but with `path='ai_query'` and the token-based prediction in `predicted_token_reference`. Notebook 04 reads the latest rows for both paths and reconciles each independently.


In [ ]:
import datetime as dt
from pyspark.sql.types import (StructType, StructField, StringType, TimestampType,
                                DoubleType, LongType)

log_schema = StructType([
    StructField("run_id",                    StringType(),    False),
    StructField("path",                      StringType(),    False),
    StructField("team",                      StringType(),    True),
    StructField("owner",                     StringType(),    True),
    StructField("run_start",                 TimestampType(), True),
    StructField("run_end",                   TimestampType(), True),
    StructField("duration_sec",              DoubleType(),    True),
    StructField("n_rows",                    LongType(),      True),
    StructField("total_chars",               LongType(),      True),
    StructField("avg_chars",                 DoubleType(),    True),
    StructField("predicted_low_list",        DoubleType(),    True),
    StructField("predicted_low_promo",       DoubleType(),    True),
    StructField("predicted_medium_list",     DoubleType(),    True),
    StructField("predicted_medium_promo",    DoubleType(),    True),
    StructField("predicted_token_reference", DoubleType(),    True),
    StructField("logged_ts",                 TimestampType(), True),
])

log_row = (
    RUN_ID, "ai_query", TEAM_TAG, OWNER_EMAIL,
    run_start, run_end,
    float((run_end - run_start).total_seconds()),
    int(PRED["n_rows"]), int(PRED["total_chars"]),
    float(PRED["total_chars"]) / max(PRED["n_rows"], 1),
    None, None, None, None,             # per-input fields: 03a's responsibility
    float(PRED["predicted_cost"]),
    utc_now(),
)

(spark.createDataFrame([log_row], schema=log_schema)
     .write
     .mode("append")
     .option("mergeSchema", "true")
     .saveAsTable(RUN_LOG_TABLE))

print(f"Logged run {RUN_ID} to {RUN_LOG_TABLE}")
display(spark.table(RUN_LOG_TABLE).orderBy(F.desc("logged_ts")).limit(5))


## Done

`ai_query` results are in `query_results`, run logged to `run_log`. Notebook 04 will reconcile both paths against `system.billing.usage` and show predicted vs. actual side-by-side.
